# 08 - Redundancy, dependencies and cascading effects

## Learning objectives

- Calculate series, parallel and k-out-of-n reliability
- Question redundancy independence assumptions
- Illustrate common-cause failure effects
- Interpret dependency and cascade graphs


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Redundancy is more than adding components

Redundancy provides the expected reliability benefit only when its channels are not defeated by the same cause. Shared power, environment, software or maintenance errors can violate independence.


In [ ]:
R=.95
print('2 series',R**2); print('2 parallel',1-(1-R)**2)


## 2. k-out-of-n

For identical independent components, $R_{k|n}=\sum_{i=k}^n {n\choose i}R^i(1-R)^{n-i}$. Example: 2oo3 voting.


In [ ]:
from math import comb
def k_out_of_n(R,k,n): return sum(comb(n,i)*R**i*(1-R)**(n-i) for i in range(k,n+1))
for r in [.8,.9,.95,.99]: print(r,k_out_of_n(r,2,3))


In [ ]:
Rs=np.linspace(.5,.999,150)
plt.plot(Rs,Rs,label='1oo1'); plt.plot(Rs,[k_out_of_n(r,1,2) for r in Rs],label='1oo2'); plt.plot(Rs,[k_out_of_n(r,2,3) for r in Rs],label='2oo3'); plt.xlabel('component R'); plt.ylabel('system R'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. Common-cause failure: a simplified mixture illustration

Use the existing teaching approximation $P_{dual}\approx\beta q+(1-\beta)q^2$: a mixture of perfectly shared and independent failure cases. This is an order-of-magnitude illustration, **not a complete standard beta-factor calculation**.


In [ ]:
q=.01
pd.DataFrame([{'beta':b,'P_dual':b*q+(1-b)*q*q,'ratio_vs_independent':(b*q+(1-b)*q*q)/(q*q)} for b in [0,.01,.05,.1,.2]])


## 4. Dependency graph

Not every dependency is a logic gate. Common power can affect several functions; communication and human dependencies can propagate downstream effects.


In [ ]:
import networkx as nx
G=nx.DiGraph([('PowerSupply','LT-101'),('PowerSupply','LSHH-101'),('PowerSupply','XV actuator'),('LT-101','HMI'),('HMI','Operator'),('LSHH-101','TripLogic'),('TripLogic','XV actuator')])
print('PowerSupply downstream:',sorted(nx.descendants(G,'PowerSupply')))
pos=nx.spring_layout(G,seed=3); nx.draw_networkx(G,pos,node_size=1800,font_size=8); plt.axis('off'); plt.show()


## 5. TK-101: common power failure

For independent component failures, automatic-shutdown unavailability is $1-(1-p_{LSHH})(1-p_{XV})$. Represent `COMMON_POWER_FAIL` affecting both as **one shared cause**, not two independent event copies.


In [ ]:
p_lshh=.01; p_xv=.005; p_common=.002
ind=1-(1-p_lshh)*(1-p_xv)
with_common=1-(1-p_common)*(1-p_lshh)*(1-p_xv)
print(ind,with_common,'relative increase',with_common/ind-1)


## 6. Cascading effects

A structural graph can illustrate the path SensorBias -> LateDetection -> HighLevelDuration -> OverflowDemand -> LossOfContainment -> IgnitionOpportunity -> FireEscalation. Probabilities and timing may require a Bayesian network, Markov model or process simulation.


In [ ]:
prop=nx.DiGraph([('SensorBias','LateDetection'),('LateDetection','HighLevelDuration'),('HighLevelDuration','OverflowDemand'),('OverflowDemand','LossOfContainment'),('LossOfContainment','IgnitionOpportunity'),('IgnitionOpportunity','FireEscalation')])
print(list(nx.topological_sort(prop)))


## Try it!

1. Compute 2oo3 reliability for R=0.90 and compare with 1oo2.
2. For q=0.001 and beta=0.01, compare dual failure with the independent case.
3. Add NetworkSwitch as a shared dependency.
4. Identify two organizational rather than purely technical TK-101 dependencies.


## Summary

Evaluate redundancy in the context of dependencies. Next, connect system risk to **maintenance strategies and decisions**.
